# Silver Population

This notebook turns the PSA DS_18 population projections into **one population per reporting unit and year** (2019–2024). It is the denominator for per-person measures (tourism intensity) and completes the Silver inputs for `fact_province_year.population_total`.

| Source | Bronze table | What it gives |
| --- | --- | --- |
| DS_18 Excel, projected mid-year population 2015–2025 | `psa_ds_18` | Province totals **including their HUCs**, NCR, City of Isabela, every year 2015–2025 |
| DS_18 PDF, by city/municipality 2020–2025 | `psa_ds_18_pdf_rows` (`04_ingest_psa_ds18_pdf`) | The 16 HUCs outside NCR, 2020–2025 |

| Silver table | One row = |
| --- | --- |
| `map_psa_population_area_to_psgc` | one DS_18 label (Excel or PDF), loaded from the reviewed seed |
| `population_row` | one DS_18 value used: source table × printed row × year, with its role and unit (traceability, reconciliation) |
| `population_unit_year` | one reporting unit per year (100 × 6) with `population_total` and `population_source` |

**Rules applied** (DL-021, schema doc #43 §7.4):

| Unit | 2019 | 2020–2024 | `population_source` |
| --- | --- | --- | --- |
| Province with no separate HUC, Quezon (Lucena already inside, DL-011), City of Isabela, NCR | Excel `Total` | Excel `Total` | `ds18_excel` |
| The 16 HUCs outside NCR | `HUC_2020 × HUC_2020 ÷ HUC_2021` | PDF row | `estimated_from_2020_trend` / `ds18_pdf` |
| The 14 provinces that contain those HUCs | Excel total − estimated HUCs | Excel total − PDF HUCs | `ds18_excel_minus_estimated_huc` / `ds18_excel_minus_huc` |
| Maguindanao del Norte, Maguindanao del Sur | NULL | NULL | NULL (DS_18 prints Maguindanao before its 2022 split) |

* Rows are found by their **label**, never by position: Excel area rows are matched to the seed and paired with the `Total` row printed right below them (`source_row_number`, DL-023); PDF HUC rows are matched within the province block they are printed in.
* Values are whole persons, as published. Only the `Total` rows are used (not age groups or Male/Female).
* Every data-quality check runs **before** writing. If a `BLOCK` check fails, results go to `dq_results`, no Silver table is written, and the notebook raises an error.

## Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid, os, re, hashlib
from datetime import datetime
import pandas as pd

# ── Unity Catalog targets ────────────────────────────────────────
CATALOG = "tuklas_dev"
BRONZE_SCHEMA = "02_bronze"
SILVER_SCHEMA = "03_silver"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "04_silver_population"

run_id = f"silver_pop_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# ── Inputs ───────────────────────────────────────────────────────
TBL_EXCEL = "psa_ds_18"              # 01_ingest_psa (needs source_row_number, DL-023)
TBL_PDF = "psa_ds_18_pdf_rows"       # 04_ingest_psa_ds18_pdf
TBL_UNIT = "geo_reporting_unit"      # 01_silver_geography
TBL_PSA_ROWS = "psa_economy_row"     # 03_silver_psa_economy (cross-check only, DQ-POP-11)

EXCEL_YEARS = list(range(2015, 2026))     # year columns DS_18 Excel must have
PDF_YEARS = list(range(2020, 2026))       # year columns DS_18 PDF must have
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]   # output years (fact covers 2019-2024)
ESTIMATE_YEAR, TREND_YEARS = 2019, (2020, 2021)   # DL-021: HUC_2019 = HUC_2020 x HUC_2020 / HUC_2021

SEED_RELATIVE_PATH = "resources/seeds/map_psa_population_area_to_psgc.csv"
SEED_PATH_OVERRIDE = None

EXPECTED_NULL_UNITS = {"1908700000": "Maguindanao del Norte", "1908800000": "Maguindanao del Sur"}
KNOWN_BOUNDARY_DIFF = {   # DQ-POP-11: units whose PSA economy area differs from the DS_18 area (DL-025)
    "1204700000": "Cotabato: DS_18 includes the 63 barangays that became the BARMM Special Geographic Area in 2024; "
                  "PSA's 2019-2023 economy tables leave them out of Cotabato",
}
REGION_TOLERANCE = 5            # DQ-POP-04 / 06: printed totals vs sum of rows (persons; PSA rounds each row)
IMPLIED_TOLERANCE_PCT = 0.5     # DQ-POP-11: our population vs the population implied by PSA's per-capita GDP

# ── Output tables ────────────────────────────────────────────────
TBL_MAP = "map_psa_population_area_to_psgc"
TBL_ROWS = "population_row"
TBL_UNIT_YEAR = "population_unit_year"

def bronze_table(name): return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{name}`"
def silver_table(name): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{name}`"
def dq_table(name): return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{name}`"

print(f"Run ID: {run_id}")

## Helpers

Same pattern as the other Silver notebooks: `dq(...)` records each check (written to `dq_results` at the end), and `write_silver(...)` overwrites a whole table.

In [0]:
dq_results = []

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

def write_silver(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table(name)))
    return spark.table(silver_table(name)).count()

def find_seed_path():
    if SEED_PATH_OVERRIDE:
        return SEED_PATH_OVERRIDE
    here = os.getcwd()
    for _ in range(8):
        candidate = os.path.join(here, SEED_RELATIVE_PATH)
        if os.path.exists(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    raise FileNotFoundError(f"{SEED_RELATIVE_PATH} not found above {os.getcwd()}. Run from the repo Git folder or set SEED_PATH_OVERRIDE.")

def text(c):
    """Trimmed text; empty cells and pandas' 'nan' / 'None' placeholders become NULL."""
    t = F.trim(c)
    return F.when(t.isNull() | t.isin("", "nan", "NaN", "None", "NaT"), None).otherwise(t)

def whole_number(c):
    """Published count as a whole number: '1,234' and '1234' and '1234.0' -> 1234; anything else -> NULL."""
    d = F.expr(f"try_cast(regexp_replace({c}, ',', '') AS DECIMAL(20,2))")
    return F.when(d == F.floor(d), d.cast("bigint"))

spark.sql(f"""
INSERT INTO {silver_table('silver_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"✓ STARTED record inserted for {run_id}")

## Step 1 — `map_psa_population_area_to_psgc`: load the DS_18 crosswalk seed

DS_18 needs its own crosswalk: the same label covers a different area than in the PSA economy tables (e.g. DS_18 `Cebu` **includes** Cebu City, Lapu-Lapu and Mandaue; the economy tables' `Cebu` excludes them).

| Seed column | Meaning |
| --- | --- |
| `source_table` | `psa_ds_18` (Excel) or `psa_ds_18_pdf_rows` (PDF) |
| `block_label` | PDF only, for HUC and region rows: the area heading the row is printed under |
| `area_label` | Row name as printed (Excel: as in the file, including the "Zambonga" spelling; PDF: mostly upper case) |
| `ds18_area` | The Excel label of the same area, so Excel and PDF rows can be compared |
| `row_role` | Excel: `unit` (loaded to the unit), `not_a_unit` (Maguindanao before the split, Cotabato City: national reconciliation only), `region_total`, `national_total`. PDF: `check_total` (must equal the Excel row; also the block headings), `region_total` (must equal the Excel region), `huc` (the HUC's population, subtracted from `from_unit_psgc_code`) |
| `unit_psgc_code`, `from_unit_psgc_code` | Unit the value belongs to; for HUC rows, the province unit it is subtracted from |

In [0]:
seed_path = find_seed_path()
seed_bytes = open(seed_path, "rb").read()
seed_sha256 = hashlib.sha256(seed_bytes).hexdigest()
seed_pd = pd.read_csv(seed_path, dtype=str, keep_default_na=False, encoding="utf-8")
seed_pd = seed_pd.apply(lambda s: s.str.strip()).replace({"": None})
pop_map = (spark.createDataFrame(seed_pd.astype(object).where(seed_pd.notna(), None),
                                 schema=", ".join(f"`{c}` STRING" for c in seed_pd.columns))
           .withColumn("_seed_file", F.lit(SEED_RELATIVE_PATH))
           .withColumn("_seed_sha256", F.lit(seed_sha256))
           .withColumn("_silver_run_id", F.lit(run_id))
           .withColumn("_processed_at", F.current_timestamp()))
print(f"Seed: {seed_path} | rows: {len(seed_pd)} | sha256: {seed_sha256[:12]}…")
display(pop_map.groupBy("source_table", "row_role").count().orderBy("source_table", "row_role"))

units = spark.table(silver_table(TBL_UNIT)).select("unit_psgc_code", "unit_name", "unit_type", "region_code", "region_name")
unit_codes = [r[0] for r in units.select("unit_psgc_code").collect()]

## Step 2 — Excel rows: one value per area and year

Bronze keeps the sheet as printed: an area name on one row, its `Total` on the next row, then age groups. The year of each column is read from the two header rows (`2019-07-01…` above `Total`), and the year columns must be exactly 2015–2025.

In [0]:
xl = spark.table(bronze_table(TBL_EXCEL))
assert "source_row_number" in xl.columns, (
    f"{TBL_EXCEL} has no source_row_number. Rerun 01_ingest_psa with the row-number change (DL-023) first.")
data_cols = sorted([c for c in xl.columns if re.fullmatch(r"_\d+", c)], key=lambda c: int(c[1:]))
xl = xl.select(F.col("source_row_number").cast("int").alias("source_row_number"),
               *[text(F.col(c)).alias(c) for c in data_cols],
               "_source_file", "_source_sha256", F.col("_run_id").alias("_bronze_run_id"))

# Header: the row with the dates and the row below it with Total / Male / Female
head = [r.asDict() for r in xl.orderBy("source_row_number").limit(10).collect()]
date_row = next(r for r in head if any(re.match(r"^(19|20)\d\d-07-01", r[c] or "") for c in data_cols))
sub_row = next(r for r in head if r["source_row_number"] == date_row["source_row_number"] + 1)
year_cols, last_year = {}, None
for c in data_cols:                     # dates sit on the Total column of each year block
    m = re.match(r"^((19|20)\d\d)-07-01", date_row[c] or "")
    last_year = int(m.group(1)) if m else last_year
    if (sub_row[c] or "") == "Total" and m:
        year_cols[int(m.group(1))] = c
assert sorted(year_cols) == EXCEL_YEARS, f"{TBL_EXCEL}: expected Total columns for {EXCEL_YEARS}, found {sorted(year_cols)}"
LABEL_COL = data_cols[0]
first_data_row = sub_row["source_row_number"] + 1
print(f"Year columns: {year_cols} | label column {LABEL_COL} | data from row {first_data_row}")

AGE_OR_TOTAL = r"^(\d+-\d+|\d+ and over|Total)$"
labels = (xl.filter((F.col("source_row_number") >= first_data_row) & F.col(LABEL_COL).isNotNull()
                    & ~F.col(LABEL_COL).rlike(AGE_OR_TOTAL))
            .select(F.col("source_row_number").alias("label_row_number"), F.col(LABEL_COL).alias("area_label")))
stack = ", ".join(f"'{y}', `{c}`" for y, c in sorted(year_cols.items()))
totals = (xl.filter(F.col(LABEL_COL) == "Total")
            .select((F.col("source_row_number") - 1).alias("label_row_number"),
                    F.col("source_row_number").cast("string").alias("source_row_number"),
                    F.expr(f"stack({len(year_cols)}, {stack}) AS (year, value_text)"),
                    "_source_file", "_source_sha256", "_bronze_run_id"))
excel_map = pop_map.filter(F.col("source_table") == TBL_EXCEL).drop("_seed_file", "_seed_sha256", "_silver_run_id", "_processed_at")

# Region each area row is printed under (by row order); region rows and NCR roll up to PHILIPPINES
w_above = Window.orderBy("label_row_number").rowsBetween(Window.unboundedPreceding, -1)
excel_labels = (labels.join(excel_map, "area_label", "left")
    .withColumn("_head", F.when(F.col("row_role").isin("region_total", "national_total"), F.col("area_label")))
    .withColumn("excel_parent_label", F.when(F.col("row_role") == "region_total", F.lit("PHILIPPINES"))
                                       .when(F.col("row_role") == "national_total", F.lit(None))
                                       .otherwise(F.last("_head", ignorenulls=True).over(w_above)))
    .drop("_head"))
excel_rows = (excel_labels.join(totals, "label_row_number", "left")
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("value", whole_number("value_text"))
    .withColumn("page_number", F.lit(None).cast("string"))
    .withColumn("pdf_line_number", F.lit(None).cast("string")))
display(excel_rows.filter(F.col("year") == 2019).groupBy("row_role").agg(F.count("*").alias("rows"), F.sum("value").alias("population_2019")))

## Step 3 — PDF rows: area totals and the 16 HUCs

Rows are read in page and line order. Each row gets the **block** it is printed in (the last area heading above it from the seed), so an HUC or region row is matched by its label *and* the block it sits in (the PDF also has a municipality named "Caraga"). Year columns must be exactly 2020–2025.

In [0]:
pdf_t = spark.table(bronze_table(TBL_PDF))
pdf_year_cols = sorted(int(c[4:]) for c in pdf_t.columns if re.fullmatch(r"pop_\d{4}", c))
assert pdf_year_cols == PDF_YEARS, f"{TBL_PDF}: expected year columns {PDF_YEARS}, found {pdf_year_cols}"

pdf_map = pop_map.filter(F.col("source_table") == TBL_PDF).drop("_seed_file", "_seed_sha256", "_silver_run_id", "_processed_at")
heads = [r[0] for r in pdf_map.filter("row_role = 'check_total'").select("area_label").collect()]
w_pdf = Window.orderBy("_page", "_line").rowsBetween(Window.unboundedPreceding, 0)
pdf_all = (pdf_t.select(text(F.col("area_label")).alias("area_label"), "page_number", "pdf_line_number",
                        F.col("page_number").cast("int").alias("_page"), F.col("pdf_line_number").cast("int").alias("_line"),
                        *[F.col(f"pop_{y}") for y in PDF_YEARS],
                        "_source_file", "_source_sha256", F.col("_run_id").alias("_bronze_run_id"))
           .withColumn("_head", F.when(F.col("area_label").isin(heads), F.col("area_label")))
           .withColumn("pdf_block", F.last("_head", ignorenulls=True).over(w_pdf)))

pdf_stack = ", ".join(f"'{y}', `pop_{y}`" for y in PDF_YEARS)
pdf_long = pdf_all.select("area_label", "pdf_block", "page_number", "pdf_line_number", "_page", "_line",
                          F.expr(f"stack({len(PDF_YEARS)}, {pdf_stack}) AS (year, value_text)"),
                          "_source_file", "_source_sha256", "_bronze_run_id")
# Area totals are unique labels; HUC and region rows are matched inside their block ("Caraga" is also a municipality)
by_label = pdf_long.join(pdf_map.filter(F.col("block_label").isNull()).drop("block_label"), "area_label")
by_block = pdf_long.join(pdf_map.filter(F.col("block_label").isNotNull()).withColumnRenamed("block_label", "pdf_block"),
                         ["area_label", "pdf_block"])
pdf_rows = (by_label.unionByName(by_block)
    .withColumnRenamed("pdf_block", "block_label")
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("value", whole_number("value_text"))
    .withColumn("source_row_number", F.lit(None).cast("string"))
    .withColumn("label_row_number", F.lit(None).cast("int"))
    .withColumn("excel_parent_label", F.lit(None).cast("string")))
display(pdf_rows.filter((F.col("row_role") == "huc") & (F.col("year") == 2020)).select("block_label", "area_label", "value").orderBy("block_label"))

## Step 4 — `population_unit_year`: one row per unit and year

* **HUCs:** the PDF value for 2020–2024; for 2019 `HUC_2020 × HUC_2020 ÷ HUC_2021`, rounded to whole persons (DL-021).
* **Provinces with HUCs:** the Excel province total minus its HUCs (estimated HUCs for 2019), so province + HUCs always equals the published total.
* **Every other unit:** the Excel `Total` as published.

`source_reference`, `_source_sha256` and `_bronze_run_id` name **every input** of a value: for a province with HUCs, the Excel total *and* each PDF HUC row subtracted from it (e.g. `psa_ds_18 row 1129 minus City of Cebu: psa_ds_18_pdf_rows p17 l9; …`).

In [0]:
cols_out = ["source_table", "ds18_area", "area_label", "block_label", "row_role", "unit_psgc_code", "from_unit_psgc_code",
            "area_psgc_code", "area_psgc_name", "excel_parent_label", "year", "value", "value_text",
            "source_row_number", "page_number", "pdf_line_number", "_source_file", "_source_sha256", "_bronze_run_id"]
rows_all = excel_rows.select(*cols_out).unionByName(pdf_rows.select(*cols_out))

base = (excel_rows.filter((F.col("row_role") == "unit") & F.col("year").isin(YEARS))
        .select("unit_psgc_code", "year", F.col("value").alias("ds18_total"),
                F.concat(F.lit(f"{TBL_EXCEL} row "), F.col("source_row_number")).alias("_ref"),
                "_source_sha256", "_bronze_run_id"))

huc_pub = (pdf_rows.filter((F.col("row_role") == "huc") & F.col("year").isin(YEARS))
           .select("unit_psgc_code", "from_unit_psgc_code", "year", "value", F.lit(False).alias("is_estimate"),
                   F.concat(F.lit(f"{TBL_PDF} p"), F.col("page_number"), F.lit(" l"), F.col("pdf_line_number")).alias("_ref"),
                   "_source_sha256", "_bronze_run_id"))
y0, y1 = TREND_YEARS
trend = (pdf_rows.filter((F.col("row_role") == "huc") & F.col("year").isin(y0, y1))
         .groupBy("unit_psgc_code", "from_unit_psgc_code", "_source_sha256", "_bronze_run_id")
         .pivot("year", [y0, y1]).agg(F.first("value")))
huc_est = trend.select("unit_psgc_code", "from_unit_psgc_code", F.lit(ESTIMATE_YEAR).alias("year"),
                       F.round(F.col(str(y0)).cast("decimal(38,6)") * F.col(str(y0)) / F.col(str(y1))).cast("bigint").alias("value"),
                       F.lit(True).alias("is_estimate"),
                       F.concat(F.lit(f"estimated {y0} x {y0} / {y1}: "), F.col(str(y0)), F.lit(" x "), F.col(str(y0)),
                                F.lit(" / "), F.col(str(y1))).alias("_ref"),
                       "_source_sha256", "_bronze_run_id")
hucs = huc_pub.unionByName(huc_est)

huc_by_province = (hucs.join(units.select(F.col("unit_psgc_code"), F.col("unit_name").alias("_huc_name")), "unit_psgc_code")
                   .groupBy(F.col("from_unit_psgc_code").alias("unit_psgc_code"), "year")
                   .agg(F.sum("value").alias("_huc_sum"), F.max("is_estimate").alias("_huc_est"),
                        F.concat_ws(", ", F.array_sort(F.collect_list("_huc_name"))).alias("_huc_names"),
                        # lineage of every HUC value subtracted (PDF rows, or the PDF rows behind a 2019 estimate)
                        F.concat_ws("; ", F.array_sort(F.collect_list(F.concat("_huc_name", F.lit(": "), "_ref")))).alias("_huc_refs"),
                        F.collect_set("_source_sha256").alias("_huc_sha"), F.collect_set("_bronze_run_id").alias("_huc_run")))

province_like = (base.join(huc_by_province, ["unit_psgc_code", "year"], "left")
    .select("unit_psgc_code", "year",
            (F.col("ds18_total") - F.coalesce(F.col("_huc_sum"), F.lit(0))).alias("population_total"),
            F.when(F.col("_huc_sum").isNull(), F.lit("ds18_excel"))
             .when(F.col("_huc_est"), F.lit("ds18_excel_minus_estimated_huc"))
             .otherwise(F.lit("ds18_excel_minus_huc")).alias("population_source"),
            F.when(F.col("_huc_sum").isNull(), F.lit("DS_18 Excel total"))
             .otherwise(F.concat(F.lit("DS_18 Excel total "), F.col("ds18_total"), F.lit(" minus "),
                                 F.when(F.col("_huc_est"), F.lit("estimated ")).otherwise(F.lit("")),
                                 F.col("_huc_names"), F.lit(" ("), F.col("_huc_sum"), F.lit(")"))).alias("population_note"),
            # every input: the Excel total and each HUC subtracted from it
            F.when(F.col("_huc_refs").isNull(), F.col("_ref"))
             .otherwise(F.concat(F.col("_ref"), F.lit(" minus "), F.col("_huc_refs"))).alias("source_reference"),
            F.array_join(F.array_sort(F.array_distinct(F.concat(F.array("_source_sha256"), F.coalesce("_huc_sha", F.array().cast("array<string>"))))), "; ").alias("_source_sha256"),
            F.array_join(F.array_sort(F.array_distinct(F.concat(F.array("_bronze_run_id"), F.coalesce("_huc_run", F.array().cast("array<string>"))))), "; ").alias("_bronze_run_id")))
huc_units = hucs.select("unit_psgc_code", "year", F.col("value").alias("population_total"),
                        F.when(F.col("is_estimate"), F.lit("estimated_from_2020_trend")).otherwise(F.lit("ds18_pdf")).alias("population_source"),
                        F.when(F.col("is_estimate"), F.lit("DS_18 PDF 2020 and 2021 projected back one year (DL-021)"))
                         .otherwise(F.lit("DS_18 PDF row")).alias("population_note"),
                        F.col("_ref").alias("source_reference"), "_source_sha256", "_bronze_run_id")
pop = province_like.unionByName(huc_units)

ugrid = units.crossJoin(spark.createDataFrame([(y,) for y in YEARS], "year INT"))
null_note = F.lit("DS_18 prints Maguindanao before its 2022 split into del Norte and del Sur; no source value (#43 7.4)")
unit_year = (ugrid.join(pop, ["unit_psgc_code", "year"], "left")
    .withColumn("population_note", F.when(F.col("unit_psgc_code").isin(list(EXPECTED_NULL_UNITS)) & F.col("population_total").isNull(), null_note)
                                    .otherwise(F.col("population_note")))
    .withColumn("_silver_run_id", F.lit(run_id))
    .withColumn("_processed_at", F.current_timestamp())
    .select("unit_psgc_code", "unit_name", "unit_type", "region_code", "region_name", "year",
            F.col("population_total").cast("bigint").alias("population_total"), "population_source", "population_note",
            "source_reference", F.lit("DS_18").alias("_source_dataset_id"), "_source_sha256", "_bronze_run_id",
            "_silver_run_id", "_processed_at"))
display(unit_year.filter(F.col("unit_name").isin("Cebu", "City of Cebu", "City of Lapu-Lapu", "City of Mandaue", "Quezon"))
        .select("unit_name", "year", "population_total", "population_source", "population_note").orderBy("unit_name", "year"))

## Step 5 — Data-quality checks

| ID | Check | Severity |
| --- | --- | --- |
| DQ-POP-01 | Every Excel area label is in the seed, and every Excel seed label is found exactly once | BLOCK |
| DQ-POP-02 | Every PDF seed row is found exactly once (HUC and region rows inside their block) | BLOCK |
| DQ-POP-03 | Every Excel area row has its `Total` row right below it | BLOCK |
| DQ-COM-06 | Every value used is a whole number | BLOCK |
| DQ-POP-04 | Excel: each region = sum of its rows, and PHILIPPINES = sum of regions + NCR (± rounding), 2015–2025 | BLOCK |
| DQ-POP-05 | Every PDF area and region total equals the Excel total, 2020–2025 (validation rule 9) | BLOCK |
| DQ-POP-06 | PDF: each province with HUCs = sum of the rows printed in its block (± rounding), so its HUCs are inside the province total | REVIEW |
| DQ-COM-03 | One row per unit and year (100 × 6) | BLOCK |
| DQ-POP-07 | Every unit-year has a population > 0, except Maguindanao del Norte / del Sur, which are NULL | BLOCK |
| DQ-POP-08 | Units + rows that are not units (Maguindanao, Cotabato City) = PHILIPPINES, every year | BLOCK |
| DQ-POP-09 | Each province with HUCs: province + its HUCs = DS_18 province total (validation rule 9) | BLOCK |
| DQ-POP-10 | The 2019 HUC estimates are listed (DL-021) | LOG |
| DQ-POP-11 | Population vs the population PSA used for its per-capita GDP (GDP × 1,000 ÷ per capita), 2019–2023, within 0.5% | REVIEW |
| DQ-COM-09 | No replacement characters (`�`) in unit names | BLOCK |

In [0]:
# DQ-POP-01: Excel labels <-> seed
unmapped = excel_labels.filter(F.col("row_role").isNull())
found = excel_labels.filter(F.col("row_role").isNotNull()).groupBy("area_label").count()
seed_bad = excel_map.join(found, "area_label", "left").filter(F.coalesce(F.col("count"), F.lit(0)) != 1)
dq("DQ-POP-01", TBL_ROWS, "BLOCK", unmapped.count() + seed_bad.count(), "every Excel label in the seed, each found once",
   f"{unmapped.count()} unmapped, {seed_bad.count()} seed labels not found exactly once",
   sample(unmapped, "label_row_number", "area_label") + sample(seed_bad, "area_label", "count"),
   notes="Add new or renamed labels to resources/seeds/map_psa_population_area_to_psgc.csv.")

# DQ-POP-02: PDF seed rows found once (one year is enough to count rows)
pdf_found = (pdf_rows.filter(F.col("year") == PDF_YEARS[0])
             .withColumn("block_label", F.when(F.col("row_role") == "check_total", "").otherwise(F.col("block_label")))
             .groupBy("row_role", "area_label", "block_label").count())
pdf_bad = (pdf_map.select("row_role", "area_label", F.coalesce("block_label", F.lit("")).alias("block_label"))
           .join(pdf_found, ["row_role", "area_label", "block_label"], "left")
           .filter(F.coalesce(F.col("count"), F.lit(0)) != 1))
dq("DQ-POP-02", TBL_ROWS, "BLOCK", pdf_bad.count(), "every PDF seed row found exactly once",
   f"{pdf_map.count()} seed rows, {pdf_bad.count()} not found exactly once", sample(pdf_bad, "block_label", "area_label", "count"))

# DQ-POP-03: each Excel area row has its Total row
no_total = excel_rows.filter(F.col("year").isNull()).select("label_row_number", "area_label").distinct()
dq("DQ-POP-03", TBL_ROWS, "BLOCK", no_total.count(), "a Total row below every area row", f"{no_total.count()} without",
   sample(no_total, "label_row_number", "area_label"))

# DQ-COM-06: whole numbers
bad_val = rows_all.filter(F.col("value").isNull())
dq("DQ-COM-06", TBL_ROWS, "BLOCK", bad_val.count(), "every value used is a whole number", f"{bad_val.count()} not",
   sample(bad_val, "source_table", "area_label", "year", "value_text"))

# DQ-POP-04: Excel region and national sums
children = (excel_rows.filter(F.col("row_role").isin("unit", "not_a_unit", "region_total"))
            .groupBy(F.col("excel_parent_label").alias("area_label"), "year").agg(F.sum("value").alias("children")))
region_chk = (excel_rows.filter(F.col("row_role").isin("region_total", "national_total"))
              .select("area_label", "year", F.col("value").alias("printed"))
              .join(children, ["area_label", "year"], "left")
              .withColumn("diff", F.col("printed") - F.coalesce(F.col("children"), F.lit(0))))
region_bad = region_chk.filter(F.abs("diff") > REGION_TOLERANCE)
dq("DQ-POP-04", TBL_ROWS, "BLOCK", region_bad.count(), f"|printed - sum of rows| <= {REGION_TOLERANCE}",
   f"{region_chk.count()} region-years checked, {region_bad.count()} outside; max |diff| {region_chk.agg(F.max(F.abs('diff'))).collect()[0][0]}",
   sample(region_bad, "area_label", "year", "diff"))

# DQ-POP-05: PDF totals = Excel totals
xl_vals = excel_rows.select("ds18_area", "year", F.col("value").alias("excel_value"))
pdf_vs_xl = (pdf_rows.filter(F.col("row_role").isin("check_total", "region_total")).select("ds18_area", "area_label", "year", F.col("value").alias("pdf_value"))
             .join(xl_vals, ["ds18_area", "year"], "left"))
pdf_xl_bad = pdf_vs_xl.filter(F.col("excel_value").isNull() | (F.col("pdf_value") != F.col("excel_value")))
dq("DQ-POP-05", TBL_ROWS, "BLOCK", pdf_xl_bad.count(), "PDF area and region totals = Excel, 2020-2025",
   f"{pdf_vs_xl.count()} area-years compared, {pdf_xl_bad.count()} differ", sample(pdf_xl_bad, "area_label", "year", "pdf_value", "excel_value"))

# DQ-POP-06: HUCs are printed inside their province total (PDF block sums)
huc_blocks = [r[0] for r in pdf_map.filter("row_role = 'huc'").select("block_label").distinct().collect()]
region_keys = pdf_rows.filter("row_role = 'region_total'").select("page_number", "pdf_line_number").distinct()
blk = pdf_all.filter(F.col("pdf_block").isin(huc_blocks)).join(region_keys, ["page_number", "pdf_line_number"], "left_anti")
blk_long = blk.select("area_label", "pdf_block", F.expr(f"stack({len(PDF_YEARS)}, {pdf_stack}) AS (year, value_text)"))
blk_long = blk_long.withColumn("value", whole_number("value_text"))
blk_head = blk_long.filter(F.col("area_label") == F.col("pdf_block")).select("pdf_block", "year", F.col("value").alias("printed"))
blk_sum = (blk_long.filter(F.col("area_label") != F.col("pdf_block")).groupBy("pdf_block", "year").agg(F.sum("value").alias("rows_sum")))
# full join: a province-year with no heading or no rows is a failure, never silently skipped
blk_chk = blk_head.join(blk_sum, ["pdf_block", "year"], "full").withColumn("diff", F.col("printed") - F.col("rows_sum"))
blk_bad = blk_chk.filter(F.col("diff").isNull() | (F.abs("diff") > REGION_TOLERANCE))
n_blk, n_blk_expected = blk_chk.count(), len(huc_blocks) * len(PDF_YEARS)
dq("DQ-POP-06", TBL_PDF, "REVIEW", blk_bad.count() + abs(n_blk_expected - n_blk),
   f"{n_blk_expected} province-years ({len(huc_blocks)} x {len(PDF_YEARS)}), |province - rows in its block| <= {REGION_TOLERANCE}",
   f"{n_blk} province-years checked, {blk_bad.count()} missing or outside; max |diff| {blk_chk.agg(F.max(F.abs('diff'))).collect()[0][0]}",
   sample(blk_bad, "pdf_block", "year", "printed", "rows_sum"))

# DQ-COM-03: grid
dup = unit_year.groupBy("unit_psgc_code", "year").count().filter("count > 1")
n_uy = unit_year.count()
dq("DQ-COM-03", TBL_UNIT_YEAR, "BLOCK", dup.count() + (0 if n_uy == len(unit_codes) * len(YEARS) else 1),
   f"{len(unit_codes)} units x {len(YEARS)} years", f"{n_uy} rows, {dup.count()} duplicates")

# DQ-POP-07: population present and positive, except the expected NULL units
exp_null = F.col("unit_psgc_code").isin(list(EXPECTED_NULL_UNITS))
pop_bad = unit_year.filter((~exp_null & (F.col("population_total").isNull() | (F.col("population_total") <= 0)))
                           | (exp_null & F.col("population_total").isNotNull()))
dq("DQ-POP-07", TBL_UNIT_YEAR, "BLOCK", pop_bad.count(), "population > 0 for every unit-year except Maguindanao del Norte / del Sur (NULL)",
   f"{pop_bad.count()} unexpected; {unit_year.filter(exp_null).count()} expected NULL", sample(pop_bad, "unit_name", "year", "population_total"))

# DQ-POP-08: units + not-a-unit rows = PHILIPPINES
nat = excel_rows.filter("row_role = 'national_total'").select("year", F.col("value").alias("national"))
other = excel_rows.filter("row_role = 'not_a_unit'").groupBy("year").agg(F.sum("value").alias("not_units"))
nat_chk = (spark.createDataFrame([(y,) for y in YEARS], "year INT")      # every output year must be checked
           .join(unit_year.groupBy("year").agg(F.sum("population_total").alias("units")), "year", "left")
           .join(other, "year", "left").join(nat, "year", "left")
           .withColumn("diff", F.col("national") - F.col("units") - F.col("not_units")))
nat_bad = nat_chk.filter(F.col("diff").isNull() | (F.col("diff") != 0))
dq("DQ-POP-08", TBL_UNIT_YEAR, "BLOCK", nat_bad.count(), "units + Maguindanao + Cotabato City = PHILIPPINES",
   f"{nat_bad.count()} years differ; 2024 national {nat_chk.filter('year = 2024').collect()[0]['national']:,}",
   sample(nat_bad, "year", "diff"))

# DQ-POP-09: province + its HUCs = DS_18 province total
prov_total = base.select("unit_psgc_code", "year", "ds18_total")
parts =(unit_year.select("unit_psgc_code", "year", "population_total")
         .join(hucs.select(F.col("unit_psgc_code"), F.col("from_unit_psgc_code").alias("prov"), "year").distinct(), ["unit_psgc_code", "year"], "left")
         .withColumn("prov", F.coalesce(F.col("prov"), F.col("unit_psgc_code")))
         .groupBy("prov", "year").agg(F.sum("population_total").alias("parts")))
huc_provs = [r[0] for r in pdf_map.filter("row_role = 'huc'").select("from_unit_psgc_code").distinct().collect()]
p9 = (parts.filter(F.col("prov").isin(huc_provs))
      .join(prov_total.withColumnRenamed("unit_psgc_code", "prov"), ["prov", "year"], "full")
      .filter(F.col("prov").isin(huc_provs) & F.col("year").isin(YEARS))
      .withColumn("diff", F.col("ds18_total") - F.col("parts")))
p9_bad = p9.filter(F.col("diff").isNull() | (F.col("diff") != 0))
n_p9, n_p9_expected = p9.count(), len(huc_provs) * len(YEARS)
dq("DQ-POP-09", TBL_UNIT_YEAR, "BLOCK", p9_bad.count() + abs(n_p9_expected - n_p9), "province + HUCs = DS_18 province total",
   f"{n_p9} of {n_p9_expected} province-years compared ({len(huc_provs)} x {len(YEARS)}), {p9_bad.count()} missing or differ",
   sample(p9_bad, "prov", "year", "diff"))

# DQ-POP-10: 2019 estimates listed
est = unit_year.filter((F.col("year") == ESTIMATE_YEAR) & (F.col("unit_type") == "HUC")).orderBy("unit_name")
dq("DQ-POP-10", TBL_UNIT_YEAR, "LOG", 0, "2019 HUC estimates listed (DL-021)", f"{est.count()} HUCs estimated",
   sample(est, "unit_name", "population_total"))

# DQ-POP-11: vs the population PSA used for its per-capita GDP
try:
    er = (spark.table(silver_table(TBL_PSA_ROWS))
          .filter(F.col("row_role").isin("unit", "add_to_unit") & F.col("measure").isin("gdp_total_current", "gdp_per_capita_current"))
          .groupBy("unit_psgc_code", "psa_label", "year").pivot("measure", ["gdp_total_current", "gdp_per_capita_current"]).agg(F.first("value"))
          .withColumn("implied", F.col("gdp_total_current") * 1000 / F.col("gdp_per_capita_current"))
          .groupBy("unit_psgc_code", "year").agg(F.sum("implied").alias("psa_implied")))
    cmp_ = (unit_year.select("unit_psgc_code", "unit_name", "year", "population_total", "population_source")
            .join(er, ["unit_psgc_code", "year"]).filter(F.col("population_total").isNotNull() & F.col("psa_implied").isNotNull())
            .withColumn("pct", F.round((F.col("population_total") - F.col("psa_implied")) / F.col("psa_implied") * 100, 3)))
    known = F.col("unit_psgc_code").isin(list(KNOWN_BOUNDARY_DIFF))
    off = cmp_.filter(~known & (F.abs("pct") > IMPLIED_TOLERANCE_PCT))
    est_cmp = cmp_.filter(F.col("population_source").isin("estimated_from_2020_trend", "ds18_excel_minus_estimated_huc"))
    max_all = cmp_.filter(~known).agg(F.max(F.abs("pct"))).collect()[0][0]
    max_est = est_cmp.agg(F.max(F.abs("pct"))).collect()[0][0]
    known_obs = "; ".join(f"{r['unit_name']} {r['year']}: {r['pct']:+.1f}%" for r in
                          cmp_.filter(known & F.col("year").isin(2019, 2023)).orderBy("unit_name", "year").collect())
    dq("DQ-POP-11", TBL_UNIT_YEAR, "REVIEW", off.count(), f"within {IMPLIED_TOLERANCE_PCT}% of PSA's implied population",
       f"{cmp_.count()} unit-years compared; max |diff| {max_all}% (2019 estimates: max {max_est}%); {off.count()} outside",
       sample(off, "unit_name", "year", "pct"),
       notes=f"Known boundary difference, not counted: {list(KNOWN_BOUNDARY_DIFF.values())[0]} ({known_obs}).")
except Exception as e:
    dq("DQ-POP-11", TBL_UNIT_YEAR, "REVIEW", 1, "PSA implied population available", f"not checked: {e}",
       notes=f"Run 03_silver_psa_economy first ({TBL_PSA_ROWS}).")

bad_chars = unit_year.filter(F.col("unit_name").contains("�"))
dq("DQ-COM-09", TBL_UNIT_YEAR, "BLOCK", bad_chars.count(), "0 unit names with �", f"{bad_chars.count()}")

dq_df = spark.createDataFrame(dq_results,
    "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
    "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING"
    ).withColumn("checked_at", F.current_timestamp())
display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))

## Step 6 — Write Silver tables and update the run log

The tables are written one at a time. If a write fails, the run log is closed as `FAILED` and lists the tables this run had already replaced, then the error is raised, so a run never stays `STARTED`.

In [0]:
dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
             "expected", "observed", "sample_keys", "checked_at", "notes") \
     .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))

block_failures = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
review_flags = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]

def update_run_log(status, tables_written, rows_written, notes):
    spark.sql(f"""
    UPDATE {silver_table('silver_run_log')}
    SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
        dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
    WHERE run_id = :run_id
    """, args={"status": status, "tw": tables_written, "rw": rows_written, "dc": len(dq_results),
               "db": len(block_failures), "dr": len(review_flags), "notes": notes, "run_id": run_id})

if block_failures:
    msg = "BLOCK checks failed: " + ", ".join(r[2] for r in block_failures) + ". No Silver tables written."
    update_run_log("FAILED", 0, 0, msg)
    raise AssertionError(msg + f" See {dq_table('dq_results')} for run_id = '{run_id}'.")

rows_out = (rows_all.filter(F.col("row_role").isNotNull())
            .withColumn("_silver_run_id", F.lit(run_id)).withColumn("_processed_at", F.current_timestamp()))
# Write one table at a time. If a write fails, the run log is closed as FAILED with the tables already
# replaced (they are from this run; the rest still hold the previous run), then the error is raised.
written = {}
try:
    for tbl, df_ in [(TBL_MAP, pop_map), (TBL_ROWS, rows_out), (TBL_UNIT_YEAR, unit_year)]:
        written[tbl] = write_silver(df_, tbl)
except Exception as e:
    done = ", ".join(written) or "none"
    update_run_log("FAILED", len(written), sum(written.values()),
                   f"Write failed on {tbl}: {str(e)[:500]}. Tables already replaced by this run: {done}.")
    raise
for tbl, n in written.items():
    print(f"✓ {silver_table(tbl)}: {n} rows")

final_status = "SUCCESS_WITH_REVIEW" if review_flags else "SUCCESS"
final_notes = (f"Silver population: {', '.join(f'{k}={v}' for k, v in written.items())}; "
               f"{len(dq_results)} checks, {len(review_flags)} for review"
               + (f" ({', '.join(r[2] for r in review_flags)})" if review_flags else "") + ".")
update_run_log(final_status, len(written), sum(written.values()), final_notes)
print(f"Run {run_id}: {final_status}\n{final_notes}")

## Preview

In [0]:
uy = spark.table(silver_table(TBL_UNIT_YEAR))
display(uy.groupBy("year").agg(F.count("population_total").alias("units_with_population"),
                               F.sum("population_total").alias("population")).orderBy("year"))
display(uy.groupBy("year", "population_source").count().orderBy("year", "population_source"))